In [ ]:
%load_ext jupyter_black

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from einops import parse_shape

In [ ]:
class TimeSeriesDataSet(Dataset):
    def __init__(
        self,
        pathfile: str,
        input_size: int = 24,
        var=["DaySin", "DayCos", "YearSin", "YearCos", "P", "T"],
        return_sequence: bool = True,
        mean: torch.Tensor | None = None,
        scale: torch.Tensor | None = None,
    ):
        """
        Data loader to infer time series.

        pathfile: path to the dataframe, csv.
        input_size: lenght of the time series used for explicative variables
        output_size: lenght of the time series to infer
        var: dictionnary of column name used for variables X and Y
        for_rnn: if the data loader is used for RNN, return also warmup Y
        """
        self.dataset = pd.read_csv(pathfile)
        self.input_size = input_size
        self.var = var
        self.return_sequence = return_sequence

        if (mean is None) and (scale is None):
            self.mean = torch.from_numpy(self.dataset[self.var].mean().values).to(
                torch.float32
            )
            self.scale = torch.from_numpy(self.dataset[self.var].std().values).to(
                torch.float32
            )
        else:
            self.mean = mean
            self.scale = scale

    def __len__(self) -> int:
        return len(self.dataset) - self.input_size - 1

    def __getitem__(self, idx) -> tuple[torch.Tensor, torch.Tensor]:
        # Get data
        X = torch.from_numpy(
            self.dataset.iloc[idx : (idx + self.input_size)][self.var].values
        ).to(torch.float32)
        if self.return_sequence:
            Y = torch.from_numpy(
                self.dataset.iloc[(idx + 1) : (idx + 1 + self.input_size)][
                    self.var
                ].values
            ).to(torch.float32)
        else:
            Y = torch.from_numpy(
                self.dataset.iloc[(idx + self.input_size + 1)][self.var].values
            ).to(torch.float32)

        # Standardize
        X = (X - self.mean) / self.scale
        Y = (Y - self.mean) / self.scale
        return X, Y

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super(RNN, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size, batch_first=True
        )
        self.fc = nn.Linear(in_features=hidden_size, out_features=input_size)
        self.hidden_size = hidden_size

    def forward(
        self,
        X: torch.Tensor,
        hidden: torch.Tensor | None = None,
        cell: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # Init hidden layers
        shapes = parse_shape(X, "b l c")
        if hidden is None:
            hidden = torch.zeros(1, shapes["b"], self.hidden_size).to(X.device)
            cell = torch.zeros_like(hidden)
        # lstm pass
        out, (hidden, cell) = self.lstm(X, (hidden, cell))
        # conversion to scalar output
        out = self.fc(out)
        return out, hidden, cell

In [ ]:
batch_size = 2048

# Init data loader
train_data = TimeSeriesDataSet("../Data/time_series/data/meteo_train.csv")
train_data_loader = DataLoader(
    train_data, batch_size=batch_size, shuffle=True, num_workers=8
)

test_data = TimeSeriesDataSet(
    "../Data/time_series/data/meteo_test.csv",
    return_sequence=False,
    mean=train_data.mean,
    scale=train_data.scale,
)
test_data_loader = DataLoader(test_data, batch_size=batch_size, num_workers=8)

valid_data = TimeSeriesDataSet(
    "../Data/time_series/data/meteo_valid.csv",
    return_sequence=False,
    mean=train_data.mean,
    scale=train_data.scale,
)
valid_data_loader = DataLoader(
    valid_data, batch_size=batch_size, shuffle=True, num_workers=8
)

In [ ]:
model = RNN(input_size=6, hidden_size=32)
# loss_fn = nn.L1Loss()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
n_epochs = 20

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = model.to(device)
valid_loss: list[float] = []
train_loss: list[float] = []

for epoch in range(n_epochs):
    model.train()
    accu = 0.0
    for X, Y in train_data_loader:
        out, hidden, cell = model(X)
        loss = loss_fn(out, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        accu += loss.item()
    train_loss.append(accu / len(train_data_loader))
    print(f"Train {epoch}: {train_loss[-1]}")

    model.eval()
    accu = 0.0
    with torch.no_grad():
        for X, Y in valid_data_loader:
            out, _, _ = model(X)
            loss = loss_fn(out[:, -1, :], Y)
            accu += loss.item()
        valid_loss.append(accu / len(valid_data_loader))
        print(f"Valid {epoch}: {valid_loss[-1]}")

In [ ]:
plt.plot(train_loss)
plt.plot(valid_loss)

In [ ]:
mean, scale = train_data.mean[-1], train_data.scale[-1]
plt.figure(figsize=(15, 5))
for s in range(48, 48 + 24):
    X, Y = test_data[s]
    out, _, _ = model(X[None, ...])
    plt.scatter(s - 48, Y[-1] * scale + mean, color="k")
    plt.scatter(s - 48, (out[0, -1, -1] * scale + mean).detach().numpy(), color="r")

# generate next days
X, Y = test_data[48]
out, hidden, cell = model(X[None, ...])
pred = [(out[0, -1, -1] * scale + mean).detach().numpy()]
for s in range(1, 24):
    out, hidden, cell = model(out)
    pred.append((out[0, -1, -1] * scale + mean).detach().numpy())

plt.scatter(range(24), pred, color="b")

In [ ]:
out.shape